In [11]:
# ========== 导入：Day 2 用本地 Ollama + 同一套抓取/摘要思路 ==========

# 导入标准库 os：路径拼接、读环境变量
import os
# 导入标准库 sys：后面用 sys.path 扩展模块搜索路径
import sys

# 把上一级目录加入模块搜索路径，以便 import 到上级目录里的 scraper
sys.path.append(os.path.abspath(".."))
# 从 dotenv 导入 load_dotenv：把 .env 密钥读进环境（本笔记本主流程走本地 Ollama，导入保留原样）
from dotenv import load_dotenv
# 从本地 scraper 导入抓取函数：把网页变成可供模型阅读的文本
from scraper import fetch_website_contents
# 从 IPython.display 导入展示工具（本格未直接调用）
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI 客户端：既可连云端，也可通过 base_url 连 Ollama 的 OpenAI 兼容接口
from openai import OpenAI


In [10]:
# ========== 探活：确认本机 Ollama 服务已启动 ==========

# 导入 requests：用 HTTP 访问本地 Ollama
import requests

# GET 本地根地址；若返回 b'Ollama is running' 说明服务正常（URL 勿改）
requests.get("http://localhost:11434").content


b'Ollama is running'

In [3]:
# 用 shell 魔法拉取本地模型 llama3.2（需已安装 ollama CLI；模型名字符串必须与后面调用一致）
!ollama pull llama3.2


]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success 


In [14]:
# 导入并创建默认 OpenAI 客户端（连云端；本笔记本后面主调用走带 base_url 的 ollama 客户端）
from openai import OpenAI
# 无参构造：密钥默认读环境变量 OPENAI_API_KEY
openai = OpenAI()


In [4]:
# ========== 角色与任务提示：与 Day 1 同一套「国会交易表格摘要」需求 ==========

# system prompt：告诉模型「你是分析国会交易的理财助手」（发给模型的指令，保留英文）
system_prompt = "You are my personal financial assistant that analyzes congress trading within a website"
# user prompt：要求近一个月、表格形式、最近交易优先（保留英文原文）
user_prompt = """
Here are the contents of a website.
Provides a short summary in table form. 
I want the data from the past one month, 
the table should show who bought what company, show the recent trades first"
"""


In [12]:
# 抓取国会交易页正文，后面会拼进 user message 的 content
website = fetch_website_contents("https://www.capitoltrades.com/trades")


In [16]:
# ========== 用 OpenAI 兼容接口调用本地 Ollama（不必改 SDK 用法） ==========

# Ollama 的 OpenAI 兼容基址：带 /v1，才能用 chat.completions 这套 API 形状
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# 创建指向本地 Ollama 的客户端；本地常见写法 api_key='ollama'（任意非空即可）
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

# 发起非流式聊天：model 用本地 llama3.2；messages = system + (user_prompt + 网页正文)
response = ollama.chat.completions.create(
    model="llama3.2", 
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt + website}
    ]
)


In [17]:
# 取出第一条 choice 的助手回复正文并打印（本地模型生成的摘要表格）
print(response.choices[0].message.content)


Based on the provided website content, I've extracted the relevant data for the past one month, with the recent trades first, and in table form. Please note that the historical data available on the website is restricted to the past 3 years.

**US Politician Stock Trades (Past 4 Weeks)**

| Politician | Traded Issuer | Traded | Filled After | Owner | Type | Size | Price |
| --- | --- | --- | --- | --- | --- | --- | --- |
| David Taylor | 3M Co (MMM:US) | buy | Feb 11, 2026 | Undisclosed | 1K–15K | $174.42 |
| David Taylor | General Motors Co (GM:US) | sell | Feb 11, 2026 | Undisclosed | 1K–15K | $34.12 |
| David Taylor | Ford Motor Co (F:US) | buy | Feb 11, 2026 | Undisclosed | 1K–15K | $23.57 |
| ... | ... | ... | ... | ... | ... | ... | ... |
| David Taylor | Alphabet Inc (GOOGL:US) | sell | Feb 27, 2026 | Undisclosed | 1K–15K | $307.38 |
| David Taylor | Amazon.com Inc (AMZN:US) | buy | Feb 26, 2026 | Undisclosed | 1K–15K | $207.92 |
| David Taylor | Google Cloud Holding (GOOGL:US) 